In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="10-personalized-news-feed",
    notebook_name="04_interview_walkthrough.ipynb"
)

# Interview Walkthrough: Personalized News Feed (45 Minutes)

## The Big Idea (For a 12-Year-Old)

Imagine you have to explain how to run a school cafeteria to a school principal in exactly 45 minutes. You have to:
1. First ask the right questions (what grades? dietary restrictions? budget?)
2. Then explain your plan clearly with drawings
3. Then defend your choices when the principal asks "but what if 500 kids show up at once?"

A machine learning system design interview is exactly like that. This notebook walks through an **entire 45-minute session** from start to finish -- what to say, when to say it, and how to handle the tricky follow-up questions.

---

## Staff-Level Technical Summary

This notebook covers:
- **Interview timeline** with phase-by-phase time allocation
- **Full interviewer/candidate dialogue** (realistic back-and-forth)
- **Whiteboard-style architecture diagrams** (retrieval -> ranking -> re-ranking funnel)
- **Live engagement score computation** (interactive demo)
- **6 common follow-up questions** with staff-level answers
- **Scoring rubric** (junior / senior / staff level expectations)
- **30-second elevator pitch** to open the interview strong
- **Key phrases cheat sheet** that interviewers love to hear

## 1. Interview Timeline

A well-structured 45-minute ML system design interview follows a consistent rhythm. Experienced candidates know how much time to spend on each phase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# === Interview Timeline Visualization ===

phases = [
    {'name': '1. Clarify\nRequirements',  'duration': 5,  'color': '#3498db',
     'goal': 'Ask 4-5 scoped questions before touching any ML'},
    {'name': '2. Frame as\nML Task',      'duration': 5,  'color': '#2ecc71',
     'goal': 'Define input/output, ML category, metrics'},
    {'name': '3. Data &\nFeatures',       'duration': 8,  'color': '#9b59b6',
     'goal': 'Raw data sources, feature engineering, affinity features'},
    {'name': '4. Model\nArchitecture',    'duration': 12, 'color': '#e67e22',
     'goal': 'Multi-task DNN: shared backbone + N heads. Justify every choice'},
    {'name': '5. Training &\nEvaluation', 'duration': 7,  'color': '#e74c3c',
     'goal': 'Loss function, offline AUC, online CTR/reaction rates'},
    {'name': '6. Serving\nArchitecture',  'duration': 5,  'color': '#1abc9c',
     'goal': 'Retrieval -> Ranking -> Re-ranking, <200ms SLA'},
    {'name': '7. Follow-up\nDiscussion',  'duration': 3,  'color': '#95a5a6',
     'goal': 'Positional bias, cold start, viral posts, retraining'},
]

total_time = sum(p['duration'] for p in phases)
assert total_time == 45

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(16, 9),
                                      gridspec_kw={'height_ratios': [2, 1.5]})

# Top: Gantt-style timeline
start = 0
y_pos = 0.5
for p in phases:
    rect = FancyBboxPatch((start, y_pos - 0.3), p['duration'] - 0.3, 0.6,
                           boxstyle='round,pad=0.05', facecolor=p['color'],
                           edgecolor='white', linewidth=2, alpha=0.9)
    ax_top.add_patch(rect)
    ax_top.text(start + p['duration']/2 - 0.15, y_pos,
                f"{p['name']}\n({p['duration']}m)",
                ha='center', va='center', fontsize=9.5, fontweight='bold', color='white')
    ax_top.text(start + p['duration']/2 - 0.15, y_pos - 0.52,
                f"{start}m", ha='center', va='center', fontsize=9, color='#333')
    start += p['duration']

ax_top.set_xlim(-0.5, 45.5)
ax_top.set_ylim(-0.2, 1.2)
ax_top.axis('off')
ax_top.set_title('45-Minute Interview Timeline', fontsize=16, fontweight='bold', pad=10)

# Bottom: pie chart of time allocation
sizes    = [p['duration'] for p in phases]
labels   = [f"{p['name'].replace(chr(10), ' ')} ({p['duration']}m)" for p in phases]
colors   = [p['color'] for p in phases]
explode  = [0.04] * len(phases)

wedges, texts, autotexts = ax_bot.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct='%1.0f%%', startangle=90, pctdistance=0.8,
    textprops={'fontsize': 9.5}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_color('white')
    at.set_fontweight('bold')
ax_bot.set_title('Time Allocation by Phase', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key timing insights:")
print("  - Model architecture gets the MOST time (12m) -- it is the heart of the answer")
print("  - Clarification gets 5m -- NEVER skip this, even if you know the answer")
print("  - Keep follow-ups brief (3m) -- signal awareness, do not go deep")

## 2. The 30-Second Elevator Pitch

The best candidates open with a crisp summary before diving into details. This signals that you have thought about the problem end-to-end and are not making it up as you go.

---

### Strong Opening (what staff-level looks like)

> "I'd design a three-stage system: retrieval fetches thousands of unseen posts from the social graph, a multi-task DNN ranks them by predicting multiple engagement types simultaneously and combining them into a weighted score, and a re-ranker applies diversity and business logic on top. The key ML insight is that user-author affinity -- how the user historically engaged with this specific author -- is the most predictive feature. For latency, the full pipeline needs to complete in under 200ms. Let me start by asking a few clarifying questions."

### Weak Opening (what junior-level looks like)

> "Um, so a news feed... we need to show posts to users. I guess we could use machine learning to rank them? Maybe a neural network? Let me think about the features..."

---

The difference: the strong candidate **leads with the solution structure**, uses **specific technical terms** (multi-task DNN, weighted engagement score, retrieval-ranking-reranking), and immediately **moves to clarification**.

## 3. Phase-by-Phase Dialogue Simulation

Here is a realistic transcript of a 45-minute session. Read both sides -- the interviewer's questions are designed to probe depth, not just surface knowledge.

In [ ]:
# === Interview Dialogue: Formatted Display ===

interview_script = [
    {
        'phase': 'PHASE 1: Clarifying Requirements (0-5 min)',
        'exchanges': [
            {
                'speaker': 'INTERVIEWER',
                'text': 'Design a personalized news feed system.',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "Before diving in, let me ask a few clarifying questions. "
                        "First -- what platform are we designing for? Facebook-style social, "
                        "Twitter-style interest graph, or LinkedIn professional content?",
                'note': '[Good: scopes the problem before assuming]',
            },
            {
                'speaker': 'INTERVIEWER',
                'text': "Let's say Facebook-style -- friend connections, groups, pages.",
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "Great. What engagement types matter? I'm thinking clicks, likes, comments, "
                        "shares -- but also negative signals like hide and block?",
                'note': None,
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'Yes, all of those plus dwell time. What else do you need to know?',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "What are the scale and latency requirements? Facebook-scale is roughly "
                        "2 billion DAU, each checking the feed maybe twice a day. And I'd guess "
                        "we need to return a ranked feed in under 200ms?",
                'note': '[Good: shows familiarity with real numbers]',
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'Exactly right. 200ms SLA, 2 billion DAU. Proceed.',
                'note': None,
            },
        ]
    },
    {
        'phase': 'PHASE 2: Framing as ML Task (5-10 min)',
        'exchanges': [
            {
                'speaker': 'CANDIDATE',
                'text': "I'll frame this as a pointwise learning-to-rank problem. "
                        "The ML objective is to maximize a weighted engagement score: "
                        "score = P(click)*1 + P(like)*5 + P(comment)*10 + P(share)*20. "
                        "We use multiple binary classifiers -- one per reaction type -- "
                        "rather than trying to maximize clicks alone, because clicks "
                        "are noisy and vulnerable to clickbait.",
                'note': '[Good: immediately frames as ML, not just engineering]',
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'Why pointwise and not pairwise or listwise?',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "At this scale -- potentially millions of candidate posts per user -- "
                        "pairwise comparisons are O(n^2) which is prohibitive. Listwise is also "
                        "expensive. Pointwise gives us O(n) scoring where we can rank each post "
                        "independently in parallel. The ranking quality loss is acceptable "
                        "compared to the compute savings.",
                'note': '[Excellent: knows trade-offs of LTR approaches]',
            },
        ]
    },
    {
        'phase': 'PHASE 3: Data and Features (10-18 min)',
        'exchanges': [
            {
                'speaker': 'CANDIDATE',
                'text': "We have four key data tables: Users, Posts, Interactions, and the Friendship graph. "
                        "Features fall into three categories. Post features: text embeddings from BERT, "
                        "image embeddings from CLIP, reaction counts, post age bucketized into <1h, 1-5h, 5-24h, 1-7d, >7d. "
                        "User features: demographics, device type, time of day. "
                        "And most importantly: user-author affinity features.",
                'note': None,
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'Tell me more about affinity features. Why are they most important?',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "According to Facebook's published research, user-author affinity is the "
                        "single most predictive signal for engagement. Concretely: like_rate "
                        "(how often the user liked this author's past posts), click_rate, "
                        "comment_rate, friendship_length_days, and the close_friend_flag. "
                        "Intuitively: you are far more likely to engage with your best friend's "
                        "post than a random page you followed two years ago.",
                'note': '[Excellent: cites specific research, gives concrete features]',
            },
        ]
    },
    {
        'phase': 'PHASE 4: Model Architecture (18-30 min)',
        'exchanges': [
            {
                'speaker': 'CANDIDATE',
                'text': "I'd use a multi-task DNN with a shared backbone and N task-specific heads. "
                        "One shared backbone -- three dense layers with BatchNorm and Dropout -- "
                        "learns representations useful for all tasks. Then separate heads for each "
                        "reaction: click, like, comment, share, skip (binary classification with sigmoid), "
                        "and dwell_time (regression with ReLU to ensure non-negative output).",
                'note': None,
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'Why not train N independent models, one per reaction type?',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "Three reasons. First, sparse reactions: shares happen less than 3% of the time. "
                        "A dedicated share model starves for training data. The multi-task backbone "
                        "transfers knowledge from data-rich tasks like clicks to data-poor tasks like shares. "
                        "Second, cost: one model to train, serve, and maintain vs N. "
                        "Third, shared representations: both click and share prediction need to understand "
                        "'what does this user find engaging?' -- the shared layers learn this once "
                        "instead of N times.",
                'note': '[Excellent: three specific reasons, not just one]',
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'How does the loss function work for multi-task learning?',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "Total loss = sum of alpha_i * L_i. Binary tasks use binary cross-entropy. "
                        "Dwell time uses Huber loss because dwell time has huge outliers -- "
                        "someone might leave the app open overnight. Huber is robust to outliers "
                        "while still giving smooth gradients near the origin. "
                        "The alpha weights are higher for rare tasks like shares to compensate for "
                        "their lower base rate in the training data.",
                'note': '[Excellent: knows WHY Huber not just WHAT Huber]',
            },
        ]
    },
    {
        'phase': 'PHASE 5: Training and Evaluation (30-37 min)',
        'exchanges': [
            {
                'speaker': 'CANDIDATE',
                'text': "For offline evaluation, I'd use ROC-AUC per task -- it measures rank ordering "
                        "quality regardless of the classification threshold, which is what matters "
                        "for ranking. For online metrics: CTR as a baseline, but more importantly "
                        "reaction rates (likes/impressions, shares/impressions) because CTR alone "
                        "is vulnerable to clickbait. Total time spent over a week captures passive "
                        "engagement. And user satisfaction surveys give the ground truth.",
                'note': '[Good: knows CTR is insufficient]',
            },
            {
                'speaker': 'INTERVIEWER',
                'text': 'How do you handle class imbalance in the training data?',
                'note': None,
            },
            {
                'speaker': 'CANDIDATE',
                'text': "Multiple approaches. Negative subsampling: keep all positives, "
                        "randomly drop negatives to get a ~1:4 ratio. Higher loss weights for "
                        "minority classes via the alpha parameters. For very imbalanced tasks "
                        "I might use focal loss, which automatically down-weights easy negatives "
                        "and focuses training on the hard examples.",
                'note': '[Good: knows multiple techniques]',
            },
        ]
    },
    {
        'phase': 'PHASE 6: Serving Architecture (37-42 min)',
        'exchanges': [
            {
                'speaker': 'CANDIDATE',
                'text': "Three-stage pipeline to hit 200ms. Retrieval: fetch all unseen posts for "
                        "the user using social graph traversal and inverted indexes -- outputs ~1000 "
                        "candidates in ~50ms. Ranking: run the multi-task DNN on all candidates, "
                        "compute engagement scores, sort -- ~100ms. Re-ranking: apply business logic "
                        "on top: diversity (no more than 3 posts from same author in a row), "
                        "sponsored content injection, content policy filters -- ~50ms.",
                'note': '[Excellent: gives specific latency budgets per stage]',
            },
        ]
    },
]

# Formatted display
INTERVIEWER_COLOR = '\033[94m'  # blue
CANDIDATE_COLOR   = '\033[92m'  # green
NOTE_COLOR        = '\033[93m'  # yellow
PHASE_COLOR       = '\033[1m'   # bold
RESET             = '\033[0m'

for section in interview_script:
    print(f"{'='*70}")
    print(f"  {section['phase']}")
    print(f"{'='*70}")
    for ex in section['exchanges']:
        if ex['speaker'] == 'INTERVIEWER':
            print(f"\n  INTERVIEWER: {ex['text']}")
        else:
            print(f"\n  CANDIDATE:   {ex['text']}")
        if ex['note']:
            print(f"  >> {ex['note']}")
    print()

## 4. Whiteboard Architecture Diagram

In a real interview you draw this on a whiteboard. Here is the equivalent in code.

In [ ]:
# === Whiteboard-Style Architecture Diagram ===

fig = plt.figure(figsize=(16, 12))
ax = fig.add_subplot(111)
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')

def draw_box(ax, x, y, w, h, label, sublabel=None, color='#3498db', fontsize=11):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                           facecolor=color, edgecolor='white', linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.15 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', color='white')
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.3, sublabel,
                ha='center', va='center', fontsize=9, color='#ecf0f1', style='italic')

def draw_arrow(ax, x1, y1, x2, y2, label=None, color='#2c3e50'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    if label:
        mx, my = (x1 + x2)/2, (y1 + y2)/2
        ax.text(mx + 0.15, my, label, fontsize=9, color=color, style='italic')

# Title
ax.text(8, 11.4, 'Personalized News Feed: System Architecture',
        ha='center', va='center', fontsize=16, fontweight='bold', color='#2c3e50')

# ---- USER ----
draw_box(ax, 6.5, 10.2, 3, 0.8, 'USER OPENS FEED', color='#7f8c8d')

# ---- DATA PIPELINE (left side) ----
ax.text(1.5, 9.6, 'DATA PIPELINE', ha='center', fontsize=10, fontweight='bold',
        color='#7f8c8d', style='italic')

data_boxes = [
    (0.2, 8.5, 2.2, 0.8, 'Users Table',        None, '#8e44ad'),
    (0.2, 7.3, 2.2, 0.8, 'Posts Table',         None, '#8e44ad'),
    (0.2, 6.1, 2.2, 0.8, 'Interactions Table',  None, '#8e44ad'),
    (0.2, 4.9, 2.2, 0.8, 'Friendship Graph',     None, '#8e44ad'),
]
for box in data_boxes:
    draw_box(ax, *box)

draw_box(ax, 0.2, 3.3, 2.2, 1.0, 'Feature Store', '(low-latency lookup)', '#6c3483')

for y in [8.9, 7.7, 6.5, 5.3]:
    draw_arrow(ax, 1.3, y, 1.3, 4.3)

# ---- PREDICTION PIPELINE (center/right) ----
ax.text(10.5, 9.6, 'PREDICTION PIPELINE (<200ms)', ha='center', fontsize=10,
        fontweight='bold', color='#7f8c8d', style='italic')

# Stage 1: Retrieval
draw_box(ax, 7.0, 8.2, 5.0, 1.2, 'STAGE 1: RETRIEVAL',
         'Social graph traversal + inverted indexes\n1B posts --> ~1000 candidates  |  <50ms',
         '#e74c3c', fontsize=12)

# Stage 2: Ranking
draw_box(ax, 7.0, 6.2, 5.0, 1.6, 'STAGE 2: RANKING',
         'Multi-task DNN (shared backbone + N heads)\nCompute engagement score per candidate\n1000 candidates --> ranked list  |  <100ms',
         '#e67e22', fontsize=12)

# Stage 3: Re-ranking
draw_box(ax, 7.0, 4.2, 5.0, 1.6, 'STAGE 3: RE-RANKING',
         'Business logic on top of ML scores\nDiversity, sponsored content, content policy\nRanked list --> final feed  |  <50ms',
         '#27ae60', fontsize=12)

# Arrows between stages
draw_arrow(ax, 9.5, 10.2, 9.5, 9.4, '~1B posts')
draw_arrow(ax, 9.5, 8.2, 9.5, 7.8, '~1000 candidates')
draw_arrow(ax, 9.5, 6.2, 9.5, 5.8, 'ranked list')
draw_arrow(ax, 9.5, 4.2, 9.5, 3.8)

# Feature store arrow into ranking
draw_arrow(ax, 2.4, 3.8, 7.0, 7.0, 'features')

# Final output
draw_box(ax, 7.0, 2.8, 5.0, 0.8, 'PERSONALIZED FEED --> USER',
         color='#2c3e50', fontsize=12)

# Multi-task model detail (bottom left)
ax.text(1.5, 2.8, 'MULTI-TASK DNN DETAIL', ha='center', fontsize=10,
        fontweight='bold', color='#7f8c8d', style='italic')

draw_box(ax, 0.2, 1.8, 2.5, 0.7, 'Shared Backbone', '256->128->64', '#2980b9', fontsize=9)
task_colors = ['#27ae60', '#8e44ad', '#e67e22', '#e74c3c', '#16a085']
task_labels = ['Click', 'Like', 'Comment', 'Share', 'Dwell']
for i, (tl, tc) in enumerate(zip(task_labels, task_colors)):
    draw_box(ax, 0.1 + i * 0.55, 0.8, 0.45, 0.7, tl, None, tc, fontsize=7)

ax.text(1.5, 0.35, 'P(click)*1 + P(like)*5 + P(cmt)*10 + P(share)*20 + ...',
        ha='center', va='center', fontsize=8.5, color='#2c3e50', fontweight='bold')

ax.text(8.0, 0.5, 'Total latency: <200ms  |  Scale: 2B DAU  |  ROC-AUC (offline) | CTR + reaction rates (online)',
        ha='center', va='center', fontsize=9, color='#555',
        bbox=dict(boxstyle='round', facecolor='#ecf0f1', edgecolor='#bdc3c7', alpha=0.9))

plt.tight_layout()
plt.show()

print("This is what you should be able to sketch on a whiteboard in ~3 minutes.")
print("Key elements the interviewer looks for:")
print("  - Three-stage serving pipeline (retrieval -> ranking -> re-ranking)")
print("  - Multi-task DNN with explicit shared backbone + task heads")
print("  - Feature store for low-latency feature lookup")
print("  - Latency budget per stage (<50ms + <100ms + <50ms = <200ms total)")

## 5. Live Engagement Score Demo

Interviewers often ask you to "walk me through how the score is actually computed for a specific example." Here is that demo.

In [ ]:
import pandas as pd

# === Interactive Engagement Score Walkthrough ===

REACTION_WEIGHTS = {
    'click':          1,
    'like':           5,
    'comment':       10,
    'share':         20,
    'friend_request':30,
    'hide':         -10,
    'block':        -50,
}

def walkthrough_engagement_score(predicted_probs, weights=REACTION_WEIGHTS, verbose=True):
    """
    Step-by-step engagement score computation -- the kind you would
    narrate aloud during a whiteboard interview.
    """
    rows = []
    total = 0.0
    for reaction, prob in predicted_probs.items():
        w = weights.get(reaction, 0)
        contrib = prob * w
        total += contrib
        rows.append({'Reaction': reaction, 'P(reaction)': prob, 'Weight': w,
                     'Contribution': contrib})
    rows.append({'Reaction': 'TOTAL ENGAGEMENT SCORE', 'P(reaction)': '',
                 'Weight': '', 'Contribution': total})
    df = pd.DataFrame(rows)
    return df, total


# Scenario: Three candidate posts, pick the winner
candidates = {
    'Post A\n(cat video)': {
        'click': 0.45, 'like': 0.35, 'comment': 0.08, 'share': 0.12,
        'friend_request': 0.00, 'hide': 0.02, 'block': 0.001
    },
    'Post B\n(news article)': {
        'click': 0.60, 'like': 0.05, 'comment': 0.02, 'share': 0.01,
        'friend_request': 0.00, 'hide': 0.10, 'block': 0.005
    },
    'Post C\n(friend\'s wedding)': {
        'click': 0.30, 'like': 0.62, 'comment': 0.28, 'share': 0.04,
        'friend_request': 0.001, 'hide': 0.01, 'block': 0.000
    },
}

scores = {}
for post_name, probs in candidates.items():
    df, total = walkthrough_engagement_score(probs, verbose=False)
    scores[post_name] = total
    print(f"{'='*55}")
    print(f"  {post_name.replace(chr(10), ' ')}: Predicted Probabilities")
    print(f"{'='*55}")
    print(df.to_string(index=False))
    print()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Score comparison
post_labels = [p.replace('\n', ' ') for p in candidates.keys()]
score_vals  = list(scores.values())
bar_colors  = ['#3498db', '#e74c3c', '#2ecc71']

bars = axes[0].bar(post_labels, score_vals, color=bar_colors, edgecolor='white', linewidth=2, width=0.6)
axes[0].set_ylabel('Total Engagement Score', fontsize=12)
axes[0].set_title('Engagement Score Comparison', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, s in zip(bars, score_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{s:.2f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

winner = max(scores, key=scores.get).replace('\n', ' ')
axes[0].set_title(f'Engagement Score Comparison\nWINNER: {winner}', fontsize=13, fontweight='bold')

# Plot 2: Score decomposition stacked bar
reactions_to_plot = ['click', 'like', 'comment', 'share', 'hide', 'block']
colors_stack = {'click': '#3498db', 'like': '#2ecc71', 'comment': '#9b59b6',
                'share': '#e67e22', 'hide': '#e74c3c', 'block': '#922b21'}

x = np.arange(len(candidates))
bottoms = np.zeros(len(candidates))
for reaction in reactions_to_plot:
    contribs = [REACTION_WEIGHTS.get(reaction, 0) * list(c.values())[i]
                for i, (post, c) in enumerate(candidates.items())
                for r, v in c.items() if r == reaction]
    # Fix: recompute cleanly
    contribs = [REACTION_WEIGHTS[reaction] * cand[reaction]
                for cand in candidates.values()]
    axes[1].bar(x, contribs, bottom=bottoms, label=f'{reaction} (w={REACTION_WEIGHTS[reaction]})',
                color=colors_stack[reaction], edgecolor='white', linewidth=0.5)
    bottoms = bottoms + np.array(contribs)

axes[1].set_xticks(x)
axes[1].set_xticklabels(post_labels, fontsize=9)
axes[1].set_ylabel('Score Contribution by Reaction', fontsize=12)
axes[1].set_title('Score Decomposition', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9, bbox_to_anchor=(1.01, 1))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Winner: {winner} (score: {scores[max(scores, key=scores.get)]:.2f})")
print("\nKey insight from this demo:")
print("  Post B has the HIGHEST click probability (0.60) but the LOWEST engagement score.")
print("  Why? Because it also has a high hide rate and very low like/share probability.")
print("  This is the clickbait problem: high CTR != high engagement quality.")

## 6. Six Common Follow-Up Questions

After you finish the main design, interviewers probe with targeted follow-ups. Here are the 6 most common ones with staff-level answers.

In [ ]:
# === Follow-Up Questions: Reference Card ===

followups = [
    {
        'q': '1. How do you handle viral posts that suddenly get millions of interactions?',
        'weak': 'Just cache them more aggressively.',
        'strong': (
            "Viral posts create a thundering-herd problem. Three mitigations:\n"
            "(a) SERVING: Add a dedicated viral post cache at the CDN layer. "
            "Once a post exceeds a virality threshold (e.g., 10k shares/hour), "
            "route its feature lookups to a pre-computed cache instead of the feature store.\n"
            "(b) RANKING: Viral posts get a global engagement boost signal that is "
            "computed in batch (every 5 minutes) rather than in real-time, to avoid "
            "thundering-herd on the recommendation service.\n"
            "(c) RE-RANKING: Cap the number of times a viral post appears across the "
            "feed for different users to prevent it from crowding out other content."
        ),
        'key_phrase': 'Thundering-herd, virality threshold, pre-computed cache, batch vs real-time',
    },
    {
        'q': '2. How do you handle cold start for new users?',
        'weak': 'Show popular posts.',
        'strong': (
            "Three-phase strategy:\n"
            "(a) ONBOARDING: Ask new users to select 3-5 interest topics during signup. "
            "These explicit signals immediately seed the ranking.\n"
            "(b) DEMOGRAPHIC FALLBACK: Use age + location to apply topic preference priors "
            "learned from similar warm users. A 17-year-old in Brazil has different "
            "priors than a 45-year-old in Japan.\n"
            "(c) EXPLORATION: In the first 30 interactions, inject a diverse mix of content "
            "and weight the engagement signal from early interactions more heavily "
            "(Thompson sampling or epsilon-greedy). After ~30 interactions, transition "
            "to full personalization as affinity features become available."
        ),
        'key_phrase': 'Three-phase: onboarding, demographic fallback, exploration; Thompson sampling',
    },
    {
        'q': '3. What is positional bias and how do you handle it?',
        'weak': 'Users click the top result more. We can fix that.',
        'strong': (
            "Positional bias means users are more likely to click posts at the top of the feed "
            "purely because of position, not quality. This creates a feedback loop: position-1 "
            "posts get many clicks -> model thinks they are high quality -> ranks them #1 -> "
            "gets even more position-inflated clicks.\n\n"
            "Two approaches:\n"
            "(a) TRAIN WITH POSITION AS FEATURE: Include feed position as an input feature "
            "during training. At inference time, set position to a default value (e.g., position=1 "
            "for all candidates). The model learns the position effect separately from quality.\n"
            "(b) INVERSE PROPENSITY WEIGHTING (IPW): Estimate P(click | position) as a baseline "
            "propensity score. Weight each training example by 1/P(click|position). "
            "This down-weights clicks from high positions (where position effect is strong) "
            "and up-weights clicks from low positions (where the user really had to seek it out)."
        ),
        'key_phrase': 'Feedback loop, position as feature, inverse propensity weighting',
    },
    {
        'q': '4. How often should you retrain the model?',
        'weak': 'Weekly.',
        'strong': (
            "It depends on the data and distribution shift rate. For a news feed:\n"
            "(a) MODEL WEIGHTS: Full retraining daily or every few days. "
            "User interests shift slowly, so weekly is often acceptable.\n"
            "(b) FEATURES: Update affinity features more frequently -- ideally hourly -- "
            "because a user who just liked 5 posts in a row signals a strong current preference "
            "that should influence the next page load.\n"
            "(c) ONLINE LEARNING: For the most time-sensitive signals (breaking news topics, "
            "viral events), consider lightweight online fine-tuning on the task heads only, "
            "while freezing the shared backbone. This is much cheaper than full retraining.\n"
            "Detection: monitor AUC/CTR on a held-out validation set daily. If it drops "
            "significantly, trigger an emergency retraining."
        ),
        'key_phrase': 'Feature freshness vs weight freshness, hourly features + daily model, online fine-tuning',
    },
    {
        'q': '5. How do you prevent the system from promoting clickbait?',
        'weak': 'Use shares not just clicks.',
        'strong': (
            "Four-layer defense:\n"
            "(a) WEIGHTED SCORE: Clicks get weight=1, shares get weight=20. "
            "Clickbait gets many clicks but few shares/likes, so it scores lower than "
            "genuinely engaging content.\n"
            "(b) POST-CLICK FEEDBACK: Measure the reaction rate AFTER clicking "
            "(did the user like the article content, or bounce immediately?). "
            "Posts where users click but immediately go back (low post-click dwell time) "
            "get penalized.\n"
            "(c) CONTENT POLICY: The re-ranking stage applies a classifier that flags "
            "content using known clickbait patterns in title/text.\n"
            "(d) LONGITUDINAL FEEDBACK: Track user satisfaction surveys. If users who "
            "see lots of a specific content type report lower satisfaction, down-weight that type."
        ),
        'key_phrase': 'Post-click dwell time, satisfaction surveys, content policy classifier',
    },
    {
        'q': '6. What about passive users who never like or comment?',
        'weak': 'Show them popular content.',
        'strong': (
            "Passive users represent a large chunk of DAU and we cannot afford to ignore them.\n"
            "Two implicit signal tasks in the multi-task DNN:\n"
            "(a) DWELL-TIME REGRESSION HEAD: Even passive users spend more time on posts they "
            "find interesting. A regression head predicts E[dwell_time] for each post. "
            "We add dwell_time * 0.1 to the engagement score (capped at 60s to avoid outliers).\n"
            "(b) SKIP BINARY HEAD: Predicts P(user scrolls past in <0.5s). High skip probability "
            "gives a negative contribution to the engagement score.\n"
            "Training data for both: collected from frontend scroll-tracking events, which are "
            "available for all users -- not just those who explicitly react."
        ),
        'key_phrase': 'Dwell-time regression, skip binary head, scroll-tracking, implicit signals',
    },
]

for fq in followups:
    print(f"{'='*70}")
    print(f"Q: {fq['q']}")
    print(f"{'='*70}")
    print(f"\nWEAK answer (junior): {fq['weak']}")
    print(f"\nSTRONG answer (staff level):")
    print(f"  {fq['strong']}")
    print(f"\nKEY PHRASES: {fq['key_phrase']}")
    print()

## 7. Scoring Rubric

Here is how interviewers actually evaluate candidates across levels.

In [ ]:
# === Scoring Rubric: What Each Level Looks Like ===

rubric = {
    'Junior (L3/L4)': {
        'color': '#e74c3c',
        'criteria': [
            ('Clarification',    'Skips or asks only 1-2 surface questions',        2),
            ('ML Framing',       'Gets to "binary classifier" but misses multi-task',3),
            ('Features',         'Lists basic features; misses user-author affinity', 2),
            ('Model',            'Mentions DNN but cannot explain shared backbone',   2),
            ('Loss Function',    'Says "cross-entropy" -- cannot explain Huber',      2),
            ('Serving',          'Mentions caching but no 3-stage pipeline detail',   2),
            ('Evaluation',       'CTR as only online metric',                         2),
            ('Follow-ups',       'Surface-level answers; one sentence each',          2),
        ],
        'pass_threshold': 50,
        'total': 50,
    },
    'Senior (L5)': {
        'color': '#e67e22',
        'criteria': [
            ('Clarification',    'Asks 4-5 targeted questions (scale, reactions, latency)', 4),
            ('ML Framing',       'Multi-task + weighted engagement score + LTR justification', 4),
            ('Features',         'All 3 categories; mentions affinity as most predictive', 4),
            ('Model',            'Shared backbone + N heads; mentions BatchNorm, Dropout', 4),
            ('Loss Function',    'Weighted sum; Huber for regression; alpha weights',      4),
            ('Serving',          '3-stage pipeline with latency budget per stage',         4),
            ('Evaluation',       'AUC offline; CTR + reaction rates + time spent online',  4),
            ('Follow-ups',       'Solid answers on cold start, positional bias, clickbait', 4),
        ],
        'pass_threshold': 62,
        'total': 80,
    },
    'Staff (L6+)': {
        'color': '#27ae60',
        'criteria': [
            ('Clarification',    'Asks with clear mental model; already plans around answers', 5),
            ('ML Framing',       'Pointwise LTR rationale; O(n) vs O(n^2) justification',     5),
            ('Features',         'Affinity with time decay; proposes feature importance experiments', 5),
            ('Model',            'Discusses expert layers (MMoE); mentions gradient conflict risk', 5),
            ('Loss Function',    'Discusses focal loss; uncertainty weighting (learned alphas)',    5),
            ('Serving',          'Discusses feature freshness, approximate nearest neighbor,\n'
                                 '                   two-tower retrieval',                        5),
            ('Evaluation',       'Mentions counterfactual evaluation; A/B test guardrails',       5),
            ('Follow-ups',       'IPW math; MMoE for positional bias; calibration discussion',    5),
        ],
        'pass_threshold': 75,
        'total': 100,
    },
}

fig, axes = plt.subplots(1, 3, figsize=(17, 7))

for ax, (level, data) in zip(axes, rubric.items()):
    categories = [c[0] for c in data['criteria']]
    max_scores = [c[2] for c in data['criteria']]
    # "Typical" scores for this level
    typical_scores = max_scores  # They are passing at this level
    
    y = np.arange(len(categories))
    bars = ax.barh(y, max_scores, color=data['color'], alpha=0.8, edgecolor='white', linewidth=1.5)
    
    for i, (cat, crit_data) in enumerate(zip(categories, data['criteria'])):
        ax.text(-0.1, i, cat, ha='right', va='center', fontsize=9, fontweight='bold')
        ax.text(crit_data[2] + 0.1, i, f"{crit_data[2]}pts",
                ha='left', va='center', fontsize=8.5, color='#555')
    
    ax.set_yticks(y)
    ax.set_yticklabels(['' for _ in categories])
    ax.set_xlabel('Max Points', fontsize=10)
    ax.set_title(f'{level}\n(pass: {data["pass_threshold"]}/{data["total"]})',
                 fontsize=12, fontweight='bold', color=data['color'])
    ax.set_xlim(0, max(max_scores) + 1.5)
    ax.grid(True, alpha=0.3, axis='x')
    ax.axvline(x=data['pass_threshold']/len(categories), color='gray',
               linestyle='--', alpha=0.5)

plt.suptitle('Interview Scoring Rubric by Level', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("=== What Separates Each Level ===")
print("\nJunior -> Senior:")
print("  Know the 3-stage serving pipeline with latency budgets")
print("  Know user-author affinity as the most predictive signal")
print("  Know WHY multi-task beats independent models (sparse data)")
print("\nSenior -> Staff:")
print("  Discuss Mixture of Experts (MMoE) for task conflict handling")
print("  Discuss two-tower retrieval for efficient ANN search")
print("  Discuss counterfactual evaluation and A/B test design")
print("  Proactively raise trade-offs instead of waiting to be asked")

## 8. Complete Cheat Sheet

Everything you need in one place. Print this out and keep it handy.

In [ ]:
# === Visual Cheat Sheet ===

fig = plt.figure(figsize=(18, 14))
ax = fig.add_subplot(111)
ax.set_xlim(0, 18)
ax.set_ylim(0, 14)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

def text_box(ax, x, y, w, h, title, items, title_color='#2c3e50', bg_color='#ecf0f1',
             fontsize=9.5):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                           facecolor=bg_color, edgecolor='#bdc3c7', linewidth=1.5, alpha=0.95)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.25, title, ha='center', va='center',
            fontsize=fontsize + 1, fontweight='bold', color=title_color)
    for i, item in enumerate(items):
        ax.text(x + 0.15, y + h - 0.55 - i * 0.38, item,
                ha='left', va='top', fontsize=fontsize - 0.5, color='#2c3e50')

# Title
ax.text(9, 13.5, 'PERSONALIZED NEWS FEED -- INTERVIEW CHEAT SHEET',
        ha='center', va='center', fontsize=17, fontweight='bold', color='#2c3e50')

# Box 1: Clarify
text_box(ax, 0.2, 11.0, 4.2, 2.2, '1. CLARIFY (5 min)',
    ['Platform: social (friends) or interest graph?',
     'Reactions: clicks, likes, shares, dwell, hide?',
     'Scale: DAU, posts/day, checks/day?',
     'Latency: 200ms SLA?',
     'Content: text, images, video?'],
    title_color='#3498db', bg_color='#ebf5fb')

# Box 2: ML Framing
text_box(ax, 4.6, 11.0, 4.2, 2.2, '2. ML FRAMING (5 min)',
    ['Pointwise LTR: score each post independently',
     'Input: (user, post) feature vector',
     'Output: P(reaction_i) for each reaction type',
     'Score = Σ P(r_i) * w_i',
     'Justify: O(n) vs O(n²) for pairwise/listwise'],
    title_color='#2ecc71', bg_color='#eafaf1')

# Box 3: Features
text_box(ax, 9.0, 11.0, 4.2, 2.2, '3. FEATURES (8 min)',
    ['Post: BERT text emb, CLIP img emb, age bucket',
     'User: demographics, device, time of day',
     '** Affinity (MOST IMPORTANT): **',
     '  like_rate, click_rate, comment_rate',
     '  friendship_days, close_friend_flag'],
    title_color='#9b59b6', bg_color='#f5eef8')

# Box 4: Model
text_box(ax, 13.4, 11.0, 4.4, 2.2, '4. MODEL (12 min)',
    ['Multi-task DNN: shared backbone + N heads',
     'Shared: 256->128->64 (BatchNorm + Dropout)',
     'Binary heads: click, like, comment, share, skip',
     'Regression head: dwell_time (ReLU output)',
     'Beats N models: sparse data + cheaper + transfer'],
    title_color='#e67e22', bg_color='#fef9e7')

# Box 5: Loss + Training
text_box(ax, 0.2, 8.5, 4.2, 2.2, '5. LOSS + TRAINING (7 min)',
    ['L_total = Σ α_i * L_i',
     'Binary: BCE  |  Regression: Huber',
     'Why Huber? Robust to dwell_time outliers',
     'Higher α for rare tasks (shares, comments)',
     'Neg subsampling: 1:4 pos:neg ratio'],
    title_color='#e74c3c', bg_color='#fdedec')

# Box 6: Serving
text_box(ax, 4.6, 8.5, 4.2, 2.2, '6. SERVING (5 min)',
    ['3-stage pipeline: <200ms total',
     'Retrieval: social graph -> ~1000 cands (<50ms)',
     'Ranking: DNN scores + engagement sort (<100ms)',
     'Re-ranking: diversity + ads + policy (<50ms)',
     'Feature store for low-latency feature lookups'],
    title_color='#1abc9c', bg_color='#e8f8f5')

# Box 7: Evaluation
text_box(ax, 9.0, 8.5, 4.2, 2.2, '7. EVALUATION',
    ['Offline: ROC-AUC per task (not accuracy!)',
     'Online: CTR (weak), reaction rates (strong)',
     'Online: total time spent (captures lurkers)',
     'Online: user satisfaction survey (gold std)',
     'CTR alone = insufficient (clickbait problem)'],
    title_color='#8e44ad', bg_color='#f5eef8')

# Box 8: Engagement Weights
text_box(ax, 13.4, 8.5, 4.4, 2.2, '8. ENGAGEMENT WEIGHTS',
    ['click          = 1   (weak, noisy)',
     'like           = 5   (explicit positive)',
     'comment        = 10  (effort invested)',
     'share          = 20  (vouches to friends)',
     'hide           = -10 | block = -50'],
    title_color='#2c3e50', bg_color='#eaecee')

# Box 9: Follow-ups (wide)
text_box(ax, 0.2, 5.8, 8.5, 2.4, '9. FOLLOW-UP QUESTIONS',
    ['Positional bias --> IPW (1/P(click|pos)) or position-as-feature, zero-out at inference',
     'Cold start     --> onboarding topics + demographic priors + exploration (30 interactions)',
     'Viral posts    --> virality cache, batch-compute viral boost, cap per-feed appearances',
     'Clickbait      --> post-click dwell, weighted score penalizes low like/share',
     'Passive users  --> dwell-time regression head + skip binary head (scroll tracking)',
     'Retraining     --> hourly features, daily model weights, online fine-tuning for breaking news'],
    title_color='#c0392b', bg_color='#fdedec')

# Box 10: Key phrases
text_box(ax, 9.0, 5.8, 8.8, 2.4, '10. KEY PHRASES (interviewers love these)',
    ['"User-author affinity is the most predictive feature per Facebook research"',
     '"Multi-task shares representations -- sparse tasks benefit from data-rich tasks"',
     '"Pointwise LTR is O(n) -- pairwise/listwise is O(n²) which is prohibitive at scale"',
     '"Huber loss is robust to dwell-time outliers (sleepers, forgotten-open apps)"',
     '"CTR alone is insufficient -- clickbait inflates CTR without increasing engagement quality"',
     '"Retrieval -> Ranking -> Re-ranking with a 200ms total SLA"'],
    title_color='#2980b9', bg_color='#ebf5fb')

# Box 11: 30-second pitch
ax.text(9, 4.8, 'THE 30-SECOND OPENING PITCH',
        ha='center', va='center', fontsize=13, fontweight='bold', color='#2c3e50')
pitch_text = (
    '"I\'d design a three-stage system: retrieval fetches candidate posts from the social graph, '
    'a multi-task DNN ranks them by predicting\n'
    'engagement probabilities across reaction types (like, share, comment, dwell) and combining them '
    'into a weighted score, and a re-ranker applies\n'
    'diversity + business logic. The key ML insight: user-author affinity is the most predictive feature. '
    'Total latency: under 200ms. Let me clarify first."'
)
rect_pitch = FancyBboxPatch((0.2, 3.5), 17.6, 1.1, boxstyle='round,pad=0.1',
                             facecolor='#2c3e50', edgecolor='#1a252f', linewidth=2, alpha=0.95)
ax.add_patch(rect_pitch)
ax.text(9, 4.05, pitch_text, ha='center', va='center', fontsize=9.5,
        color='white', style='italic')

plt.tight_layout()
plt.savefig('/tmp/news_feed_cheat_sheet.png', dpi=150, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()

print("Cheat sheet saved to /tmp/news_feed_cheat_sheet.png")

In [ ]:
# === Full Text Cheat Sheet (printable) ===

cheat_sheet = """
╔══════════════════════════════════════════════════════════════════════════════╗
║       PERSONALIZED NEWS FEED -- COMPLETE INTERVIEW CHEAT SHEET              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1. CLARIFY                                                                  ║
║     - Platform: social (friends) or interest graph?                          ║
║     - Reactions: clicks, likes, shares, dwell, hide, block?                  ║
║     - Scale: 2B DAU, 2 checks/day, 200ms SLA                                 ║
║                                                                              ║
║  2. ML OBJECTIVE                                                             ║
║     Score = P(click)*1 + P(like)*5 + P(comment)*10 + P(share)*20            ║
║     Pointwise LTR: score each (user, post) pair independently [O(n)]         ║
║                                                                              ║
║  3. FEATURES (in order of importance)                                        ║
║     ** User-Author Affinity **  <-- MOST PREDICTIVE (Facebook research)      ║
║        like_rate, click_rate, comment_rate, friendship_days, close_friend    ║
║     Post: BERT text emb, CLIP img emb, reaction counts, age bucket           ║
║     User: demographics, device type, time of day                             ║
║                                                                              ║
║  4. MODEL: MULTI-TASK DNN                                                    ║
║     Shared backbone (256->128->64) + N task heads                            ║
║     Binary heads: click, like, comment, share, skip [sigmoid]                ║
║     Regression head: dwell_time [ReLU, non-negative]                         ║
║     Why vs N independent? Sparse data + cheaper + transfer learning          ║
║                                                                              ║
║  5. LOSS: L_total = Σ α_i * L_i                                              ║
║     Binary: BCE  |  Dwell time: Huber (outlier-robust)                       ║
║     Higher α for rare tasks (comment, share)                                 ║
║     Negative subsampling: 1:4 positive:negative ratio                        ║
║                                                                              ║
║  6. SERVING: 3-stage pipeline (<200ms total)                                 ║
║     Retrieval:  social graph -> ~1000 candidates       <50ms                 ║
║     Ranking:    multi-task DNN -> engagement scores   <100ms                 ║
║     Re-ranking: diversity + ads + policy filters       <50ms                 ║
║                                                                              ║
║  7. METRICS                                                                  ║
║     Offline: ROC-AUC per task                                                ║
║     Online:  CTR (weak!), reaction rates, time spent, satisfaction surveys   ║
║                                                                              ║
║  8. ADVANCED TOPICS                                                          ║
║     Positional bias:  IPW or position-as-feature (zero at inference)         ║
║     Cold start:       onboarding + demographic priors + exploration           ║
║     Viral posts:      virality cache + batch-compute boost                   ║
║     Clickbait:        post-click dwell + weighted score + content policy      ║
║     Passive users:    dwell-time head + skip head (scroll-tracking data)      ║
║     Retraining:       hourly features, daily weights, online fine-tuning      ║
║                                                                              ║
║  OPENING PITCH (memorize this):                                              ║
║  "Three-stage system: retrieval -> multi-task DNN ranking                    ║
║   (weighted engagement score) -> re-ranking. Key insight: user-author        ║
║   affinity is the most predictive feature. 200ms SLA. Let me clarify first."║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

print(cheat_sheet)

## Key Takeaways

1. **Structure beats content**: A well-paced, organized 45-minute answer beats a technically brilliant but rambling one. Follow the 6-phase structure.

2. **Open with a pitch, not a question**: Staff-level candidates give a 30-second structure overview before clarifying -- it signals you already know the solution shape.

3. **Lead with the most important things first**: In features, user-author affinity is first. In the model, say "multi-task DNN" before the architecture details.

4. **Know WHY, not just WHAT**: Huber because of outliers. Multi-task because sparse data. Pointwise because O(n). These justifications distinguish senior from junior.

5. **CTR is never the only metric**: Always mention reaction rates and total time spent. Interviewers explicitly listen for this.

6. **Raise advanced topics proactively**: Positional bias, cold start, and clickbait are the three most common follow-ups. Mention them briefly in your main answer before being asked.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)